# Week 13: Data Poisoning, Model Evasion, and Final Project Evidence

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST2412_Data_Security_Privacy_Ethics/blob/main/week_13/week13_data_poisoning_evasion_colab.ipynb)

This notebook is for Week 13 of CST 2412. It connects **data poisoning** and **model evasion** to the final project.

You are not building a real security product today. You are practicing three beginner-friendly analyst habits:

1. Check whether a dataset looks trustworthy enough to use.
2. Test how bad or manipulated data can change an analysis.
3. Explain limits and recommendations without overclaiming.

The notebook uses the small Week 12 class CSV from GitHub, so it should run in Colab without uploading anything.

## How This Connects to Week 13

Week 13 is about **Data Poisoning and Model Evasion**.

Plain-language definitions:

- **Data poisoning** means the data going into a system is corrupted, manipulated, mislabeled, incomplete, biased, or otherwise misleading.
- **Model evasion** means a system can be fooled at the moment it is used. A risky event may be shaped so it does not trigger a rule, threshold, filter, or model.
- **False positive** means the system flags something normal as risky.
- **False negative** means the system misses something risky.
- **Threshold** means the cutoff where a system changes its decision, such as `score >= 32` means `needs review`.

Useful research anchors:

- NIST Adversarial Machine Learning taxonomy: https://www.nist.gov/publications/adversarial-machine-learning-taxonomy-and-terminology-attacks-and-mitigations
- NIST AI Risk Management Framework: https://www.nist.gov/itl/ai-risk-management-framework
- OWASP Machine Learning Security Top 10: https://owasp.org/www-project-machine-learning-security-top-10/
- OWASP ML02 Data Poisoning Attack: https://owasp.org/www-project-machine-learning-security-top-10/docs/ML02_2023-Data_Poisoning_Attack

## What You Will Do

Run the notebook from top to bottom.

You will:

1. Load the Week 12 security metrics CSV from GitHub.
2. Build a simple priority score.
3. Simulate a data poisoning problem.
4. Compare the clean result with the poisoned result.
5. Test how threshold choices can create false positives and false negatives.
6. Draft final-project language about evidence, limits, and recommendations.

## Step 1: Import Tools

`pandas` helps us work with tables.

`matplotlib` helps us make a simple chart.

Both are already available in Google Colab.

In [ ]:
# pandas is the main Python library we use for table-shaped data.
# We give it the nickname pd because that is the common convention.
import pandas as pd

# matplotlib is a common Python charting library.
# We give pyplot the nickname plt because that is the common convention.
import matplotlib.pyplot as plt

# This makes pandas show more columns when we display a table.
# Without this, pandas may hide columns if the table is wide.
pd.set_option("display.max_columns", 50)

# This makes chart text a little easier to read in Colab.
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

## Step 2: Load the Class CSV From GitHub

This notebook loads the CSV directly from the course GitHub repository.

That means students do not need to upload the file manually.

In [ ]:
# This is the raw GitHub URL for the Week 12 class dataset.
# A raw URL gives Python the actual CSV content instead of the normal GitHub web page.
csv_url = "https://raw.githubusercontent.com/lolusername/CST2412_Data_Security_Privacy_Ethics/main/week_12/day_1/data/week12_security_metrics.csv"

# pd.read_csv reads a CSV file and turns it into a pandas DataFrame.
# A DataFrame is like a spreadsheet table inside Python.
df = pd.read_csv(csv_url)

# .head() shows the first five rows so we can quickly check that the file loaded correctly.
df.head()

## Step 3: Inspect the Dataset Before Trusting It

Before analyzing a dataset, an analyst should ask basic trust questions:

- How many rows are there?
- What columns exist?
- Are there missing values?
- Do the column names make sense?
- Does the dataset describe people, systems, organizations, or sensitive activity?

This is not advanced coding. It is basic evidence checking.

In [ ]:
# .shape returns two numbers: number of rows and number of columns.
# Example: (10, 12) means 10 rows and 12 columns.
print("Rows and columns:", df.shape)

# .columns lists the column names.
# list(...) makes the result easier to read.
print("Column names:")
print(list(df.columns))

# .isna() checks every cell for missing data.
# .sum() counts how many missing values appear in each column.
print()
print("Missing values by column:")
print(df.isna().sum())

In [ ]:
# .describe() gives summary statistics for numeric columns.
# This helps us notice strange minimums, maximums, or averages.
df.describe()

## Step 4: Make a Simple Priority Score

This is **not** a real security scoring system.

It is a classroom example that shows how analysts turn several columns into a simple decision rule.

The score will use:

- asset criticality
- whether the system is internet exposed
- whether it stores sensitive data
- failed login count
- successful logins after failures
- MFA denials
- malware alerts
- highest alert severity

The important lesson is that any scoring rule depends on assumptions. If the data is wrong, the score can become wrong.

In [ ]:
def yes_no_to_number(series):
    """Convert a yes/no text column into 1/0 numbers."""

    # pandas works best with numbers when we calculate scores.
    # yes becomes 1 because the risk factor is present.
    # no becomes 0 because the risk factor is not present.
    # Missing or unexpected values become 0 in this beginner example.
    return (
        series
        .astype(str)                 # make sure every value is text first
        .str.lower()                 # make Yes, YES, and yes behave the same
        .map({"yes": 1, "no": 0})   # convert known labels into numbers
        .fillna(0)                   # replace unknown labels with 0
    )


def add_priority_score(input_df):
    """Return a copy of the dataset with classroom priority-score columns added."""

    # .copy() prevents us from accidentally changing the original DataFrame.
    scored = input_df.copy()

    # Convert text categories into numbers.
    # These mappings are judgment calls, not universal facts.
    scored["criticality_points"] = (
        scored["asset_criticality"]
        .astype(str)
        .str.lower()
        .map({"low": 1, "medium": 2, "high": 3})
        .fillna(0)
    )

    scored["severity_points"] = (
        scored["highest_alert_severity"]
        .astype(str)
        .str.lower()
        .map({"low": 1, "medium": 2, "high": 3})
        .fillna(0)
    )

    # Convert yes/no columns into 1/0 numbers.
    scored["internet_points"] = yes_no_to_number(scored["internet_exposed"])
    scored["sensitive_data_points"] = yes_no_to_number(scored["sensitive_data"])

    # Build a simple score.
    # The weights below are for class practice only.
    # They let us see how changing data changes the final ranking.
    scored["priority_score"] = (
        scored["criticality_points"] * 8
        + scored["internet_points"] * 4
        + scored["sensitive_data_points"] * 6
        + scored["failed_logins_24h"] * 0.25
        + scored["successful_logins_after_failures"] * 5
        + scored["mfa_denials"] * 1.5
        + scored["malware_alerts"] * 8
        + scored["severity_points"] * 4
    )

    # A threshold is a cutoff.
    # Here, a score of 32 or higher means the system should be reviewed first.
    scored["needs_review"] = scored["priority_score"] >= 32

    return scored


# Apply the scoring function to the original dataset.
clean_scores = add_priority_score(df)

# Show the most important output columns, sorted from highest priority to lowest priority.
clean_scores[[
    "asset_id", "system_name", "business_unit", "priority_score", "needs_review", "notes"
]].sort_values("priority_score", ascending=False)

## Step 5: Visualize the Clean Scores

A chart can make a ranking easier to explain in a final project.

Remember: charts can look convincing even when the data is weak. The visual is only as trustworthy as the evidence behind it.

In [ ]:
# Sort systems from lowest score to highest score so the bar chart is easy to read.
plot_data = clean_scores.sort_values("priority_score", ascending=True)

# Create a horizontal bar chart.
plt.barh(plot_data["system_name"], plot_data["priority_score"])

# Draw the review threshold as a vertical dashed line.
plt.axvline(32, color="red", linestyle="--", label="Review threshold")

# Add a title and labels so the chart makes sense outside the notebook.
plt.title("Week 13 Example: Clean Priority Scores")
plt.xlabel("Priority score")
plt.ylabel("System")
plt.legend()
plt.show()

## Step 6: Simulate Data Poisoning

Now we will create a **poisoned copy** of the dataset.

This does not attack anything. It is a safe classroom simulation.

The idea is simple: if important values are changed, removed, mislabeled, or made less severe, the analysis may produce a different result.

In this example, the poisoned version makes two risky systems look less risky.

In [ ]:
# Start with a copy so the original clean data stays unchanged.
poisoned_df = df.copy()

# Example poisoning change 1:
# Make the Student Portal look less suspicious by lowering failed logins,
# removing successful logins after failures, and lowering alert severity.
poisoned_df.loc[poisoned_df["asset_id"] == "S001", "failed_logins_24h"] = 4
poisoned_df.loc[poisoned_df["asset_id"] == "S001", "successful_logins_after_failures"] = 0
poisoned_df.loc[poisoned_df["asset_id"] == "S001", "mfa_denials"] = 1
poisoned_df.loc[poisoned_df["asset_id"] == "S001", "highest_alert_severity"] = "low"

# Example poisoning change 2:
# Make the Finance File Share look less serious by removing the malware alert
# and lowering the severity label.
poisoned_df.loc[poisoned_df["asset_id"] == "S004", "malware_alerts"] = 0
poisoned_df.loc[poisoned_df["asset_id"] == "S004", "highest_alert_severity"] = "low"

# Add a note so anyone reading the data knows this is a classroom simulation.
poisoned_df.loc[
    poisoned_df["asset_id"].isin(["S001", "S004"]),
    "notes"
] = "CLASSROOM POISONING SIMULATION: important risk signals were reduced"

# Score the poisoned dataset using the same function as before.
poisoned_scores = add_priority_score(poisoned_df)

poisoned_scores[[
    "asset_id", "system_name", "priority_score", "needs_review", "notes"
]].sort_values("priority_score", ascending=False)

## Step 7: Compare Clean Data Against Poisoned Data

This is the key Week 13 lesson.

The scoring rule did not change. The code did not change. The only thing that changed was the input data.

If the input data is poisoned, incomplete, or misleading, the final output can also become misleading.

In [ ]:
# Select the columns we need from the clean scored dataset.
clean_compare = clean_scores[[
    "asset_id", "system_name", "priority_score", "needs_review"
]].rename(columns={
    "priority_score": "clean_score",
    "needs_review": "clean_needs_review"
})

# Select the same columns from the poisoned scored dataset.
poisoned_compare = poisoned_scores[[
    "asset_id", "priority_score", "needs_review"
]].rename(columns={
    "priority_score": "poisoned_score",
    "needs_review": "poisoned_needs_review"
})

# .merge() joins two tables using a shared column.
# Here, asset_id lets us compare each system to itself.
comparison = clean_compare.merge(poisoned_compare, on="asset_id")

# Calculate how much the score changed.
comparison["score_change"] = comparison["poisoned_score"] - comparison["clean_score"]

# Identify whether the review decision changed.
comparison["review_decision_changed"] = (
    comparison["clean_needs_review"] != comparison["poisoned_needs_review"]
)

# Sort by the biggest negative score change.
comparison.sort_values("score_change")

In [ ]:
# Focus on the two systems we intentionally changed.
changed = comparison[comparison["asset_id"].isin(["S001", "S004"])]

# Make a side-by-side bar chart for clean score vs poisoned score.
changed.set_index("system_name")[["clean_score", "poisoned_score"]].plot(kind="bar")

plt.axhline(32, color="red", linestyle="--", label="Review threshold")
plt.title("Clean Scores vs. Poisoned Scores")
plt.ylabel("Priority score")
plt.xlabel("System")
plt.xticks(rotation=20, ha="right")
plt.legend()
plt.show()

## Reflection: Data Poisoning

Answer these in your lab document or notes:

1. Which system changed the most after poisoning?
2. Did the review decision change for any system?
3. Why is this a security issue?
4. Why is this a privacy or ethics issue?
5. What would you write as a limitation if you used this dataset in a final project?

## Step 8: Thresholds and Model Evasion

A threshold is a cutoff.

Thresholds are useful because they make decisions simple. But they can also create problems:

- If the threshold is too low, the system may create too many false positives.
- If the threshold is too high, the system may miss risky cases and create false negatives.
- If someone understands the threshold, they may try to stay just below it.

That last idea connects to **model evasion**: a system may be fooled at the moment it is used.

In [ ]:
# Test several possible thresholds.
# This helps us see how one cutoff changes the number of systems marked for review.
threshold_results = []

for threshold in [25, 32, 40, 50]:
    # Count how many systems would be reviewed at this threshold.
    review_count = (clean_scores["priority_score"] >= threshold).sum()

    # Add one row to our results list.
    threshold_results.append({
        "threshold": threshold,
        "systems_marked_for_review": review_count,
        "systems_not_marked_for_review": len(clean_scores) - review_count
    })

# Convert the list of dictionaries into a DataFrame.
threshold_df = pd.DataFrame(threshold_results)
threshold_df

In [ ]:
# Find systems close to the original threshold.
# These are important because small data changes could move them across the cutoff.
threshold = 32

borderline = clean_scores.copy()
borderline["distance_from_threshold"] = (borderline["priority_score"] - threshold).abs()

borderline[[
    "asset_id", "system_name", "priority_score", "needs_review", "distance_from_threshold", "notes"
]].sort_values("distance_from_threshold").head(5)

## Step 9: Simulate an Evasion Problem

In a real system, evasion can be much more complex.

For this beginner class example, evasion means a risky pattern is made to look less obvious to a simple rule.

Example: If a review process relies heavily on failed login counts, an attacker might spread attempts across time or accounts so the count stays below the alert threshold.

Again, this is a safe classroom simulation. Do not test real systems.

In [ ]:
# Make another copy for an evasion simulation.
evasion_df = df.copy()

# In the original data, the Email System has several strong risk signals.
# In this simulation, imagine the visible signals were reduced enough to look less alarming.
evasion_df.loc[evasion_df["asset_id"] == "S009", "failed_logins_24h"] = 9
evasion_df.loc[evasion_df["asset_id"] == "S009", "successful_logins_after_failures"] = 0
evasion_df.loc[evasion_df["asset_id"] == "S009", "mfa_denials"] = 2
evasion_df.loc[evasion_df["asset_id"] == "S009", "highest_alert_severity"] = "medium"
evasion_df.loc[evasion_df["asset_id"] == "S009", "notes"] = "CLASSROOM EVASION SIMULATION: visible signals were reduced"

# Score the evasion dataset.
evasion_scores = add_priority_score(evasion_df)

# Compare the Email System before and after the evasion simulation.
email_clean = clean_scores[clean_scores["asset_id"] == "S009"][[
    "asset_id", "system_name", "priority_score", "needs_review", "notes"
]]

email_evasion = evasion_scores[evasion_scores["asset_id"] == "S009"][[
    "asset_id", "system_name", "priority_score", "needs_review", "notes"
]]

pd.concat([
    email_clean.assign(version="clean data"),
    email_evasion.assign(version="evasion simulation")
])

## Reflection: Model Evasion and Limits

Answer these in your lab document or notes:

1. What signal did the example reduce?
2. Did the system still deserve attention even after the visible signals changed?
3. What could create a false negative in this kind of analysis?
4. What could create a false positive?
5. What recommendation would be realistic without overclaiming?

## Step 10: Create a Small Summary Table for a Final Project

A final project does not need to be complicated.

A useful coding project can produce a small table, explain what it means, and discuss limits.

In [ ]:
# Create a small final-project-style summary table.
summary = clean_scores[[
    "asset_id",
    "system_name",
    "business_unit",
    "priority_score",
    "needs_review",
    "internet_exposed",
    "sensitive_data",
    "highest_alert_severity"
]].sort_values("priority_score", ascending=False)

# .head(5) keeps only the top five rows.
# This is useful if you want to discuss the highest-priority systems first.
top_five = summary.head(5)
top_five

In [ ]:
# Save the summary table as a CSV file.
# In Colab, the file will appear in the file browser on the left.
top_five.to_csv("week13_priority_summary.csv", index=False)

print("Saved week13_priority_summary.csv")

## Final Project Writing Starter

Use this structure to turn notebook work into final project writing.

Do not copy this word-for-word. Replace it with your own details.

### Evidence
I used a small security metrics dataset with columns such as asset criticality, internet exposure, sensitive data, failed logins, MFA denials, malware alerts, and alert severity. I used these columns to create a simple priority score for deciding which systems should be reviewed first.

### Finding
The highest-priority systems were not simply the systems with the most failed logins. The strongest concerns appeared when several risk signals appeared together, such as high asset criticality, sensitive data, internet exposure, successful logins after failures, MFA denials, or malware alerts.

### Limitation
One limitation is that the score depends on the accuracy of the input data and the scoring weights. If login counts, severity labels, or malware alert fields were missing, manipulated, or misunderstood, the ranking could become misleading. This project should therefore be read as a classroom analysis, not as proof that any real system is safe or unsafe.

### Recommendation
A realistic recommendation is to review systems where multiple risk signals appear together instead of relying on one signal by itself. This can reduce false positives from normal activity and reduce false negatives where risky activity is hidden by a single low-looking number.

### Privacy and Ethics
The dataset describes systems rather than named people, which lowers privacy risk. However, real security logs can include account names, IP addresses, locations, timestamps, and behavior patterns. A careful analyst should limit access, avoid unnecessary personal details, and explain what the data can and cannot prove.

## Student Checklist

Before you leave Week 13, make sure you can answer these for your final project:

- What evidence or dataset am I using?
- Who created the evidence?
- What could make the evidence misleading?
- What is one false positive risk?
- What is one false negative risk?
- What is one limitation I should admit?
- What is one realistic recommendation I can defend?